## LLM Usage
I have used LLMs in order to ask questions, have a better understanding, getting access to relevant documentation in order to solve the assignment and to get improvement ideas for the code.
### Personal details
Full Name: Jarl Dang <br>
Civic Number: 20010827-9191




## Dependencies
First pip install the following libraries; torch, transformers, peft
> pip install torch, transformers, peft

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

## Process and load the data
First we need to process and load the data. As it was given, each review comes as a line together with a classifier at the start of the line. In order to tokenize it we first seperate the text and the classifier, thereafter we simply split the text on blank spaces.

In [ ]:

def load_data(filepath):
    texts = []
    labels = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                labels.append(int(parts[0]))
                texts.append(parts[1])
    return texts, labels


train_texts, train_labels = load_data("data/ReviewBaseTraining.txt")
val_texts, val_labels = load_data("data/ReviewBaseValidation.txt")
test_texts, test_labels = load_data("data/ReviewBaseTest.txt")

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"Test samples: {len(test_texts)}")


## Setting up the model and applying LoRA
We specify the model name to be deberta-v3, since that is the version we want to use according to the assignment. There after we config the model using LoRA according to the documentation on kaggle and huggingface. This allows us to only fine tune a small number of parameters instead of all of them, immensly saving on computational power and time. The values chosen are standard values which are common for these kind of projects.

<br>
Then we tokenize the data in a way fitting for the deberta model using their tokenizer.

In [ ]:

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# https://www.kaggle.com/code/verifyitisyou/fine-tuning-deberta-large-with-lora-adapter
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)


## Dataset Wrapper
PyTorch's `DataLoader` requires data to be wrapped in a `Dataset` object. This class wraps the tokenized encodings and labels into a format compatible with PyTorch. The `__getitem__` method converts each sample's encodings and label into tensors on demand, while filtering out `token_type_ids` since DeBERTa-v3 does not use them. We then create dataset instances for training, validation, and test data.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
            if key != "token_type_ids"
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)
